In [78]:
import inspect
from collections.abc import Callable
from dataclasses import dataclass

import dishka
import dishka.plotter

In [79]:
class Provider(dishka.Provider):
    def __init__(self):
        super().__init__()
        self.register()

    def __init_subclass__(cls, scope: dishka.BaseScope = dishka.Scope.APP) -> None:
        super().__init_subclass__()
        cls.scope = scope

    def register(self):
        methods = [
            method
            for _, method in inspect.getmembers(self, inspect.ismethod)
            if method.__func__.__qualname__.startswith(f"{self.__class__.__name__}.")
        ]
        for method in methods:
            self.provide(method, scope=self.scope)

In [80]:
CONTROLLER_META = "__controller__"
ROUTE_META = "__route__"


@dataclass
class Route:
    method: str
    path: str
    handler: Callable


class Controller:
    prefix: str = ""

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)

        if "prefix" not in cls.__dict__:
            raise TypeError(f"{cls.__name__} must define 'prefix'")

    # def __init_subclass__(cls, prefix: str) -> None:
    #     cls.prefix = prefix

    @classmethod
    def get_routes(cls):
        routes = []

        for _, method in inspect.getmembers(cls, inspect.isfunction):
            route = getattr(method, "__route__", None)

            if route:
                routes.append(
                    Route(
                        method=route.method,
                        path=cls.prefix + route.path,
                        handler=method,
                    )
                )

        return routes

In [81]:
class Module:
    imports: tuple[type["Module"], ...]
    providers: tuple[type["Provider"], ...]
    controllers: tuple[type["Controller"], ...]
    exports: tuple[type[Provider], ...]

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)

        if missing := (
            {"imports", "providers", "controllers", "exports"} - cls.__dict__.keys()
        ):
            raise TypeError(f"{cls.__name__} must define {missing}")

    def get_providers(self):
        providers: list[Provider] = []

        for module_cls in self.imports:
            module = module_cls()
            providers.extend(module.get_providers())

        providers.extend(provider() for provider in self.providers)

        return providers

    def get_routes(self):
        routes: list[Route] = []
        for controller in self.controllers:
            routes.extend(controller.get_routes())

        return routes

In [82]:
def get(path=""):
    def decorator(handler):
        handler.__route__ = Route(method="GET", path=path, handler=handler)
        return handler

    return decorator


@dataclass(frozen=True)
class AppConfig:
    environment: str


class UserService:
    def __init__(self, config: AppConfig):
        self.config = config

    def list_users(self) -> list[str]:
        return ["Ada", "Grace"]


class AuditService:
    def __init__(self, user_service: UserService):
        self.user_service = user_service

    def report(self) -> str:
        return f"{len(self.user_service.list_users())} users audited"


class ConfigProvider(Provider):
    def app_config(self) -> AppConfig:
        return AppConfig(environment="test")


class UserProvider(Provider):
    def user_service(self, config: AppConfig) -> UserService:
        return UserService(config)


class AuditProvider(Provider, scope=dishka.Scope.REQUEST):
    def audit_service(self, user_service: UserService) -> AuditService:
        return AuditService(user_service)


class UserController(Controller):
    prefix = "/users"

    @get("/")
    def list_users(self) -> list[str]:
        return ["route registered"]


class AppModule(Module):
    imports = ()
    providers = (ConfigProvider, UserProvider, AuditProvider)
    controllers = (UserController,)
    exports = (UserProvider, AuditProvider)

In [83]:
def create(app_module: type[Module]):
    module = app_module()
    routes = module.get_routes()
    container = dishka.make_container(*module.get_providers())

    config = container.get(AppConfig)
    user_service = container.get(UserService)
    # audit_service = container.get(AuditService)

    assert config.environment == "test"
    assert user_service.config is config
    # assert audit_service.user_service is user_service
    # assert audit_service.report() == "2 users audited"
    assert [(route.method, route.path) for route in routes] == [("GET", "/users/")]

    print("Routes:", [(route.method, route.path) for route in routes])
    print("Environment:", config.environment)
    # print("Audit:", audit_service.report())
    print("Arbre de dépendances:")
    print(dishka.plotter.render_mermaid(container))


create(AppModule)

Routes: [('GET', '/users/')]
Environment: test
Arbre de dépendances:
<html>
<head>
    <meta charset="UTF-8">
</head>
<body>

<pre class="mermaid">
---
  config:
    class:
      hideEmptyMembersBox: true
---
classDiagram
direction LR
namespace Scope.APP {
class factory3["📥 Container"]{
 
}
class factory4["🏭 AppConfig"]{
ConfigProvider.app_config()
}
class factory5["🏭 UserService"]{
UserProvider.user_service()
AppConfig
}
}

factory4 <.. factory5
namespace Scope.REQUEST {
class factory8["📥 Container"]{
 
}
class factory9["🏭 AuditService"]{
AuditProvider.audit_service()
UserService
}
}

factory5 <.. factory9

</pre>

<script type="module">
    import mermaid
    from 'https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs';
    mermaid.initialize(config);
</script>
</body>
</html>


